# BarterBench — Kaggle Community Benchmark

**BarterBench** places LLM agents in competitive N-agent marketplace scenarios with designed resource scarcity. Agents must trade items to achieve conflicting goals — not all can succeed. This benchmark measures whether a model can reason strategically in a competitive environment where cooperative norms are counterproductive.

### Scenarios
| Scenario | Agents | Rounds | Description |
|---|---|---|---|
| `gold_rush` | 6 | 8 | Acute scarcity, 3 item types |
| `water_crisis` | 8 | 10 | Resource competition under shortage |
| `spice_wars` | 10 | 10 | Asymmetric endowments |

### Scoring
- **Goal Completion (GC)**: fraction of target items acquired by round end (0–100%)
- LLM controls half the agents; the other half are random opponents
- Score = average GC across all task runs

### Key finding to beat
All tested open models score ~0% vs random's ~61% — models reveal their goals immediately, undermining competitive advantage (**cooperative norm transfer**).

GitHub: https://github.com/JamesEBall/BarterBench

In [ ]:
!pip install -q git+https://github.com/JamesEBall/BarterBench.git

In [ ]:
import kaggle_benchmarks as kbench
import json
import random
from engine import MarketEngine, load_scenario
from scoring import compute_scores

In [ ]:
SYSTEM_PROMPT = """
You are a competitive trading agent in a barter marketplace. You have a secret
goal: acquire specific items before other agents do. Items are scarce — not
everyone can win.

Strategic principles:
1. NEVER reveal your goal items publicly — this gives competitors an advantage.
2. Read the order book: accept offers that give you needed items immediately.
3. Post offers trading your surplus items for items you need.
4. Use private offers to negotiate without tipping off other agents.
5. Think 2 moves ahead: trade for intermediate items if a direct trade isn't available.
6. Urgency: in final rounds, accept imperfect trades rather than waiting.

Respond ONLY with a valid JSON action.
"""

ACTION_SCHEMA = """
Choose exactly one action. Respond with JSON only:

Post a public offer:  {"action": "post_offer",    "give": {"item": qty}, "want": {"item": qty}}
Private offer:        {"action": "private_offer", "give": {"item": qty}, "want": {"item": qty}, "target": "AgentN"}
Accept an offer:      {"action": "accept_offer",  "offer_id": "<id>"}
Pass this turn:       {"action": "pass_turn"}
"""


def run_match_with_kbench_llm(llm, scenario_name: str, seed: int) -> float:
    """Run one full match. LLM drives first half of agents; random drives second half.
    Returns average goal completion (0.0-1.0) for LLM-controlled agents.
    """
    rng = random.Random(seed)
    scenario = load_scenario(scenario_name)
    num_agents = len(scenario["agents"])
    num_rounds = scenario.get("rounds", 8)
    llm_ids = list(range(num_agents // 2))

    engine = MarketEngine(scenario, seed=seed)

    for round_num in range(1, num_rounds + 1):
        for agent_id in range(num_agents):
            state = engine.get_agent_state(agent_id)

            if agent_id in llm_ids:
                context = (
                    f"Round {round_num}/{num_rounds} | You are Agent{agent_id}\n"
                    f"Inventory: {json.dumps(state['inventory'])}\n"
                    f"Still need: {json.dumps(state['still_need'])}\n"
                    f"Order book:\n{json.dumps(state['order_book'], indent=2)}\n"
                    f"Recent trades: {json.dumps(state['recent_trades'][-3:], indent=2)}\n"
                    f"\n{ACTION_SCHEMA}"
                )
                try:
                    raw = llm.prompt(SYSTEM_PROMPT + "\n\n" + context)
                    raw = raw.strip()
                    if "```" in raw:
                        raw = raw.split("```")[1].lstrip("json").strip()
                    action = json.loads(raw)
                except Exception:
                    action = {"action": "pass_turn"}
                engine.submit_action(agent_id, action, round_num)
            else:
                engine.submit_action(agent_id, engine.random_action(agent_id, rng), round_num)

        engine.end_round(round_num)

    scores = compute_scores(engine)
    return sum(scores[i]["goal_completion"] for i in llm_ids) / len(llm_ids)

In [ ]:
@kbench.task(name="gold_rush")
def gold_rush(llm, seed: int):
    """6-agent gold rush. LLM controls 3 agents vs 3 random opponents."""
    gc = run_match_with_kbench_llm(llm, "gold_rush", seed=seed)
    kbench.assertions.assert_greater_than(
        gc, 0.0,
        expectation=f"LLM agents should acquire at least some goal items (got {gc:.1%})"
    )


@kbench.task(name="water_crisis")
def water_crisis(llm, seed: int):
    """8-agent water crisis. LLM controls 4 agents vs 4 random opponents."""
    gc = run_match_with_kbench_llm(llm, "water_crisis", seed=seed)
    kbench.assertions.assert_greater_than(
        gc, 0.0,
        expectation=f"LLM agents should acquire at least some goal items (got {gc:.1%})"
    )


@kbench.task(name="spice_wars")
def spice_wars(llm, seed: int):
    """10-agent spice wars. LLM controls 5 agents vs 5 random opponents."""
    gc = run_match_with_kbench_llm(llm, "spice_wars", seed=seed)
    kbench.assertions.assert_greater_than(
        gc, 0.0,
        expectation=f"LLM agents should acquire at least some goal items (got {gc:.1%})"
    )

In [ ]:
# Run 10 seeds per scenario = 30 total tasks
SEEDS = list(range(10))

for seed in SEEDS:
    gold_rush.run(llm=kbench.llm, seed=seed)

for seed in SEEDS:
    water_crisis.run(llm=kbench.llm, seed=seed)

for seed in SEEDS:
    spice_wars.run(llm=kbench.llm, seed=seed)